In [40]:
import os
import pandas as pd

raw_path = os.path.join("..", "data", "raw", "01_fund_master.csv")
df_master = pd.read_csv(raw_path)

df_master['launch_date'] = pd.to_datetime(df_master['launch_date'])

processed_folder = os.path.join("..", "data", "processed")
output_path = os.path.join(processed_folder, "01_fund_master_cleaned.csv")

df_master.to_csv(output_path, index=False)
print("File cleaned and saved successfully!")


File cleaned and saved successfully!


In [41]:
import os
import pandas as pd

nav_path = os.path.join("..", "data", "raw", "02_nav_history.csv")
df_nav = pd.read_csv(nav_path)

print(df_nav.info())
print(df_nav.isnull().sum())

<class 'pandas.DataFrame'>
RangeIndex: 46000 entries, 0 to 45999
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   amfi_code  46000 non-null  int64  
 1   date       46000 non-null  str    
 2   nav        46000 non-null  float64
dtypes: float64(1), int64(1), str(1)
memory usage: 1.1 MB
None
amfi_code    0
date         0
nav          0
dtype: int64


In [42]:
import os
import pandas as pd

raw_dir = os.path.join("..", "data", "raw")
proc_dir = os.path.join("..", "data", "processed")
os.makedirs(proc_dir, exist_ok=True)

# 1. Clean NAV History
df_nav = pd.read_csv(os.path.join(raw_dir, "02_nav_history.csv"))
df_nav['date'] = pd.to_datetime(df_nav['date'])
df_nav.to_csv(os.path.join(proc_dir, "02_nav_history_cleaned.csv"), index=False)
print("Saved: 02_nav_history_cleaned.csv")

# 2. Clean AUM by Fund House
df_aum = pd.read_csv(os.path.join(raw_dir, "03_aum_by_fund_house.csv"))
df_aum['date'] = pd.to_datetime(df_aum['date'])
df_aum.to_csv(os.path.join(proc_dir, "03_aum_by_fund_house_cleaned.csv"), index=False)
print("Saved: 03_aum_by_fund_house_cleaned.csv")

# 3. Clean Benchmark Indices
df_bench = pd.read_csv(os.path.join(raw_dir, "10_benchmark_indices.csv"))
df_bench['date'] = pd.to_datetime(df_bench['date'])
df_bench.to_csv(os.path.join(proc_dir, "10_benchmark_indices_cleaned.csv"), index=False)
print("Saved: 10_benchmark_indices_cleaned.csv")

# 4. Copy other files that do not need date cleaning directly to processed
simple_files = [
    "04_monthly_sip_inflows.csv",
    "05_category_inflows.csv",
    "06_industry_folio_count.csv",
    "07_scheme_performance.csv",
    "08_investor_transactions.csv",
    "09_portfolio_holdings.csv"
]

for file in simple_files:
    raw_path = os.path.join(raw_dir, file)
    if os.path.exists(raw_path):
        df_temp = pd.read_csv(raw_path)
        df_temp.to_csv(os.path.join(proc_dir, file.replace(".csv", "_cleaned.csv")), index=False)
        print(f"Saved: {file.replace('.csv', '_cleaned.csv')}")

Saved: 02_nav_history_cleaned.csv
Saved: 03_aum_by_fund_house_cleaned.csv
Saved: 10_benchmark_indices_cleaned.csv
Saved: 04_monthly_sip_inflows_cleaned.csv
Saved: 05_category_inflows_cleaned.csv
Saved: 06_industry_folio_count_cleaned.csv
Saved: 07_scheme_performance_cleaned.csv
Saved: 08_investor_transactions_cleaned.csv
Saved: 09_portfolio_holdings_cleaned.csv


In [43]:
import os
import sqlite3
import pandas as pd
import numpy as np

db_dir = os.path.join("..", "data", "db")
os.makedirs(db_dir, exist_ok=True)
db_path = os.path.join(db_dir, "mutual_funds.db")
conn = sqlite3.connect(db_path)

proc_dir = os.path.join("..", "data", "processed")
tables = {
    "01_fund_master_cleaned.csv": "fund_master",
    "02_nav_history_cleaned.csv": "nav_history",
    "03_aum_by_fund_house_cleaned.csv": "aum_by_fund_house",
    "04_monthly_sip_inflows_cleaned.csv": "monthly_sip_inflows",
    "05_category_inflows_cleaned.csv": "category_inflows",
    "06_industry_folio_count_cleaned.csv": "industry_folio_count",
    "07_scheme_performance_cleaned.csv": "scheme_performance",
    "08_investor_transactions_cleaned.csv": "investor_transactions",
    "09_portfolio_holdings_cleaned.csv": "portfolio_holdings",
    "10_benchmark_indices_cleaned.csv": "benchmark_indices"
}

for file_name, table_name in tables.items():
    file_path = os.path.join(proc_dir, file_name)
    if os.path.exists(file_path):
        df = pd.read_csv(file_path)
        df.to_sql(table_name, conn, if_exists="replace", index=False)
        print(f"Loaded Table: {table_name}")

df_nav = pd.read_sql_query("SELECT amfi_code, date, nav FROM nav_history ORDER BY amfi_code, date", conn)
df_nav['date'] = pd.to_datetime(df_nav['date'])

analytics_data = []
for amfi_code, group in df_nav.groupby('amfi_code'):
    group = group.sort_values('date')
    start_nav = group.iloc[0]['nav']
    end_nav = group.iloc[-1]['nav']
    start_date = group.iloc[0]['date']
    end_date = group.iloc[-1]['date']
    
    years = (end_date - start_date).days / 365.25
    if years > 0 and start_nav > 0:
        cagr = (end_nav / start_nav) ** (1 / years) - 1
    else:
        cagr = 0.0
        
    group['daily_return'] = group['nav'].pct_change()
    daily_vol = group['daily_return'].std()
    annual_vol = daily_vol * np.sqrt(252) if not pd.isna(daily_vol) else 0.0
    
    analytics_data.append({
        "amfi_code": amfi_code,
        "start_nav": start_nav,
        "end_nav": end_nav,
        "years": round(years, 2),
        "cagr_pct": round(cagr * 100, 2),
        "annual_volatility_pct": round(annual_vol * 100, 2)
    })

df_analytics = pd.DataFrame(analytics_data)
df_analytics.to_sql("fund_analytics", conn, if_exists="replace", index=False)
conn.close()

print("Success: Database loaded and Advanced Metrics calculated!")

Loaded Table: fund_master
Loaded Table: nav_history
Loaded Table: aum_by_fund_house
Loaded Table: monthly_sip_inflows
Loaded Table: category_inflows
Loaded Table: industry_folio_count
Loaded Table: scheme_performance
Loaded Table: investor_transactions
Loaded Table: portfolio_holdings
Loaded Table: benchmark_indices
Success: Database loaded and Advanced Metrics calculated!


In [44]:
import os
import sqlite3
import pandas as pd

db_path = os.path.join("..", "data", "db", "mutual_funds.db")
conn = sqlite3.connect(db_path)

df_analytics = pd.read_sql_query("SELECT * FROM fund_analytics", conn)
processed_folder = os.path.join("..", "data", "processed")
output_path = os.path.join(processed_folder, "11_fund_analytics_cleaned.csv")

df_analytics.to_csv(output_path, index=False)
conn.close()

print("Successfully exported 11_fund_analytics_cleaned.csv!")

Successfully exported 11_fund_analytics_cleaned.csv!


In [45]:
import os
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

db_path = os.path.join("..", "data", "db", "mutual_funds.db")
conn = sqlite3.connect(db_path)

df_analytics = pd.read_sql_query("SELECT * FROM fund_analytics", conn)
df_master = pd.read_sql_query("SELECT amfi_code, scheme_name, category, risk_category FROM fund_master", conn)
df_sip = pd.read_sql_query("SELECT month, sip_inflow_crore FROM monthly_sip_inflows", conn)

df_merged = pd.merge(df_analytics, df_master, on="amfi_code")

output_dir = os.path.join("..", "dashboard")
os.makedirs(output_dir, exist_ok=True)

sns.set_theme(style="whitegrid")

# Chart 1: Risk vs Return Scatter Plot
plt.figure(figsize=(10, 6))
sns.scatterplot(
    data=df_merged, 
    x="annual_volatility_pct", 
    y="cagr_pct", 
    hue="risk_category", 
    size="years", 
    sizes=(50, 200),
    palette="viridis"
)
plt.title("Risk vs. Return (Volatility vs. CAGR)", fontsize=14, fontweight="bold")
plt.xlabel("Annual Volatility (%)", fontsize=12)
plt.ylabel("CAGR (%)", fontsize=12)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.savefig(os.path.join(output_dir, "01_risk_vs_return.png"), dpi=300)
plt.close()

# Chart 2: Average CAGR by Category
plt.figure(figsize=(10, 5))
avg_cagr = df_merged.groupby("category")["cagr_pct"].mean().reset_index()
sns.barplot(data=avg_cagr, x="category", y="cagr_pct", palette="Blues_r")
plt.title("Average CAGR (%) by Fund Category", fontsize=14, fontweight="bold")
plt.xlabel("Category", fontsize=12)
plt.ylabel("Average CAGR (%)", fontsize=12)
plt.tight_layout()
plt.savefig(os.path.join(output_dir, "02_cagr_by_category.png"), dpi=300)
plt.close()

# Chart 3: Monthly SIP Inflows Trend
plt.figure(figsize=(12, 5))
sns.lineplot(data=df_sip, x="month", y="sip_inflow_crore", marker="o", color="teal", linewidth=2)
plt.title("Monthly SIP Inflows Trend", fontsize=14, fontweight="bold")
plt.xlabel("Month", fontsize=12)
plt.ylabel("Inflow (Crores)", fontsize=12)
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(os.path.join(output_dir, "03_sip_inflows_trend.png"), dpi=300)
plt.close()

conn.close()
print("Success: All 3 professional analytical charts saved as images in the 'dashboard/' folder!")

C:\Users\HP\AppData\Local\Temp\ipykernel_12096\1820284439.py:43: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(data=avg_cagr, x="category", y="cagr_pct", palette="Blues_r")


Success: All 3 professional analytical charts saved as images in the 'dashboard/' folder!
